<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [2]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [3]:
import folium
import pandas as pd

In [4]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [4]:
launch_sites = [
    {
        "name": "CCAFS SLC-40",
        "latitude": 28.561857,
        "longitude": -80.577366
    },
    {
        "name": "KSC LC-39A",
        "latitude": 28.608058,
        "longitude": -80.603956
    },
    {
        "name": "VAFB SLC-4E",
        "latitude": 34.632093,
        "longitude": -120.610829
    },
    {
        "name": "Omelek Island",
        "latitude": 9.047721,
        "longitude": 167.743129
    }
]

launch_sites_df = pd.DataFrame(launch_sites)
launch_sites_df

,name,latitude,longitude
0,CCAFS SLC-40,28.561857,-80.577366
1,KSC LC-39A,28.608058,-80.603956
2,VAFB SLC-4E,34.632093,-120.610829
3,Omelek Island,9.047721,167.743129


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [5]:
spacex_map = folium.Map(
    location=[25, -40],
    zoom_start=3
)

for _, site in launch_sites_df.iterrows():
    folium.Marker(
        location=[site["latitude"], site["longitude"]],
        popup=site["name"],
        tooltip=site["name"]
    ).add_to(spacex_map)

spacex_map

In [6]:
# Download and read the `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)

Now, you can take a look at what are the coordinates for each site.


In [7]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [8]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [9]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [10]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


In [11]:
# Create a new map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Add launch site circles
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    folium.Circle(
        location=coordinate,
        radius=1000,
        popup=row['Launch Site'],
        color='blue',
        fill=True
    ).add_to(site_map)

# Create a marker cluster
marker_cluster = MarkerCluster().add_to(site_map)

# Mark each launch as success or failure
for index, row in spacex_df.iterrows():
    coordinate = [row['Lat'], row['Long']]

    if row['class'] == 1:
        color = 'green'
    else:
        color = 'red'

    folium.Marker(
        location=coordinate,
        icon=folium.Icon(color=color),
        popup=f"Launch Site: {row['Launch Site']} | Class: {row['class']}"
    ).add_to(marker_cluster)

# Display the map
site_map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [12]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [13]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [14]:
spacex_df['marker_color'] = spacex_df['class'].apply(
    lambda x: 'green' if x == 1 else 'red'
)

In [15]:
spacex_df[['class', 'marker_color']].tail(10)

,class,marker_color
46,1,green
47,1,green
48,1,green
49,1,green
50,1,green
51,0,red
52,0,red
53,0,red
54,1,green
55,0,red


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [16]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed, 
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']
for index, record in spacex_df.iterrows():
    # TODO: Create and add a Marker cluster to the site map
    # marker = folium.Marker(...)
    marker_cluster.add_child(marker)

site_map

In [17]:
for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(
            color='white',
            icon_color=record['marker_color'],
            icon='info-sign'
        ),
        popup=record['Launch Site']
    )
    marker_cluster.add_child(marker)

site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [18]:
# Add Mouse Position to the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"

MousePosition(
    position='topright',
    separator=' | ',
    empty_string='Unavailable',
    lng_first=False,
    num_digits=20,
    prefix='Coordinates:',
    lat_formatter=formatter,
    lng_formatter=formatter
).add_to(site_map)

site_map


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [19]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [20]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [21]:
launch_site_lat = 28.573255
launch_site_lon = -80.646895

coastline_lat = 28.56367
coastline_lon = -80.57163

distance_coastline = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    coastline_lat,
    coastline_lon
)

distance_coastline

7.42932194658424

In [22]:
# Coordinates of the closest coastline point
coastline_lat = 28.56367
coastline_lon = -80.57163

coordinate = [coastline_lat, coastline_lon]

# Create a marker to display the distance
distance_marker = folium.Marker(
    coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>'
        % "{:10.2f} KM".format(distance_coastline)
    )
)

# Add the marker to the map
site_map.add_child(distance_marker)

# Display the map
site_map

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [23]:
# Create a PolyLine between the launch site and coastline
coordinates = [
    [launch_site_lat, launch_site_lon],
    [coastline_lat, coastline_lon]
]

lines = folium.PolyLine(
    locations=coordinates,
    weight=1
)

site_map.add_child(lines)

site_map

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [24]:
nearby_lat = 28.56367
nearby_lon = -80.57163

distance_nearby = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    nearby_lat,
    nearby_lon
)

distance_marker = folium.Marker(
    [nearby_lat, nearby_lon],
    icon=DivIcon(
        icon_size=(150, 30),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>{:.2f} KM</b></div>'.format(distance_nearby)
    )
)

site_map.add_child(distance_marker)

nearby_coordinates = [
    [launch_site_lat, launch_site_lon],
    [nearby_lat, nearby_lon]
]

lines = folium.PolyLine(
    locations=nearby_coordinates,
    weight=2,
    color='purple'
)

site_map.add_child(lines)
site_map

In [25]:
import folium

# Create a global map
global_map = folium.Map(
    location=[20, 0],
    zoom_start=2
)

# Add all launch site markers
for index, row in launch_sites_df.iterrows():
    folium.Marker(
        location=[row['Lat'], row['Long']],
        popup=row['Launch Site'],
        tooltip=row['Launch Site'],
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(global_map)

# Display the map
global_map

In [26]:
# Create a map centered around the launch sites
site_map = folium.Map(
    location=[28.5, -80.6],
    zoom_start=5
)

# Add circles for all launch sites
for index, row in launch_sites_df.iterrows():
    folium.Circle(
        location=[row['Lat'], row['Long']],
        radius=1000,
        popup=row['Launch Site'],
        color='blue',
        fill=True
    ).add_to(site_map)

# Create a marker cluster
marker_cluster = MarkerCluster().add_to(site_map)

# Add color-coded markers for each launch
for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        popup=f"Launch Site: {record['Launch Site']} | Class: {record['class']}",
        icon=folium.Icon(
            color='white',
            icon_color=record['marker_color'],
            icon='info-sign'
        )
    )
    marker_cluster.add_child(marker)

# Display the map
site_map

In [27]:
import folium
from folium.features import DivIcon

# Launch site coordinates
launch_site_coordinate = [launch_site_lat, launch_site_lon]

# Coastline coordinates
coastline_coordinate = [coastline_lat, coastline_lon]

# Calculate distance
distance_coastline = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    coastline_lat,
    coastline_lon
)

# Create a zoomed-in map
proximity_map = folium.Map(
    location=launch_site_coordinate,
    zoom_start=12,
    width=1600,
    height=900
)

# Mark the launch site
folium.Marker(
    launch_site_coordinate,
    popup="SpaceX Launch Site",
    tooltip="Launch Site",
    icon=folium.Icon(color="blue")
).add_to(proximity_map)

# Mark the coastline point and display distance
folium.Marker(
    coastline_coordinate,
    tooltip="Selected Coastline Point",
    icon=DivIcon(
        icon_size=(150, 30),
        icon_anchor=(0, 0),
        html=(
            '<div style="font-size:14px; color:#d35400; '
            'background-color:white;"><b>{:.2f} km</b></div>'
        ).format(distance_coastline)
    )
).add_to(proximity_map)

# Draw a line between launch site and coastline
folium.PolyLine(
    locations=[launch_site_coordinate, coastline_coordinate],
    weight=3,
    color="purple",
    tooltip=f"Distance: {distance_coastline:.2f} km"
).add_to(proximity_map)

# Save and display
proximity_map.save("spacex_proximity_map.html")
proximity_map

In [29]:
import folium

# १. लाँच साईट आणि जवळच्या ठिकाणांचे कॉर्डीनेट्स (Coordinates) निश्चित करा
launch_site_lat, launch_site_lon = 28.562302, -80.577356  # CCAFS SLC 40
highway_lat, highway_lon = 28.56335, -80.57085           # Highway
railway_lat, railway_lon = 28.57205, -80.58528           # Railway
coast_lat, coast_lon = 28.56235, -80.56784               # Coastline

# २. बेस मॅप तयार करा (मॅप लाँच साईटवर सेंटर होईल)
site_map = folium.Map(location=[launch_site_lat, launch_site_lon], zoom_start=15)

# ३. लाँच साईटसाठी लाल रंगाचा मार्कर (Red Marker) जोडा
folium.Marker(
    [launch_site_lat, launch_site_lon],
    popup="CCAFS SLC 40 Launch Site",
    icon=folium.Icon(color='red', icon='rocket', prefix='fa')
).add_to(site_map)

# ४. जवळच्या ठिकाणांचे मार्कर्स आणि अंतर दर्शवणारी निळी रेषा (PolyLine) जोडणे

# अ) समुद्रकिनारा (Coastline) - 0.93 km
folium.Marker(
    [coast_lat, coast_lon],
    popup="Coastline: 0.93 km",
    icon=folium.Icon(color='blue', icon='info-sign')
).add_to(site_map)

folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [coast_lat, coast_lon]], 
    weight=2.5, 
    color='blue'
).add_to(site_map)

# ब) हायवे (Highway) - 0.65 km
folium.Marker(
    [highway_lat, highway_lon],
    popup="Highway: 0.65 km",
    icon=folium.Icon(color='green', icon='info-sign')
).add_to(site_map)

folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [highway_lat, highway_lon]], 
    weight=2.5, 
    color='green'
).add_to(site_map)

# क) रेल्वे (Railway) - 1.34 km
folium.Marker(
    [railway_lat, railway_lon],
    popup="Railway: 1.34 km",
    icon=folium.Icon(color='purple', icon='info-sign')
).add_to(site_map)

folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [railway_lat, railway_lon]], 
    weight=2.5, 
    color='purple'
).add_to(site_map)

# ५. मॅप डिस्प्ले करा
site_map


In [5]:
import folium

# 1. Coordinate bounds for the 3 unique SpaceX launch sites
launch_sites = {
    'CCAFS SLC 40': [28.562302, -80.577356],
    'KSC LC 39A': [28.573255, -80.646895],
    'VAFB SLC 4E': [34.632834, -120.610745]
}

# 2. Initialize a base map centered over the United States
# We use a lower zoom level (zoom_start=4) to show the overall global distribution clearly
global_map = folium.Map(location=[31.0, -100.0], zoom_start=4)

# 3. Use a loop to add markers for each launch site with interactive pop-ups
for site_name, coordinates in launch_sites.items():
    folium.Marker(
        location=coordinates,
        popup=f"<b>Launch Site:</b> {site_name}<br><b>Status:</b> Active Operational",
        icon=folium.Icon(color='blue', icon='plane', prefix='fa')
    ).add_to(global_map)

# 4. Display the generated global map overview
global_map


In [6]:
import folium
from folium.plugins import MarkerCluster

# 1. Initialize the base map centered over the United States
outcome_map = folium.Map(location=[31.0, -100.0], zoom_start=4)

# 2. Sample records representing the launch locations and their outcomes from your dataset
# Class 1 = Success (Green Marker), Class 0 = Failure (Red Marker)
launch_records = [
    {'site': 'CCAFS SLC 40', 'coordinates': [28.562302, -80.577356], 'class': 1, 'flight': 'Flight 5'},
    {'site': 'CCAFS SLC 40', 'coordinates': [28.562302, -80.577356], 'class': 0, 'flight': 'Flight 13'},
    {'site': 'KSC LC 39A', 'coordinates': [28.573255, -80.646895], 'class': 1, 'flight': 'Flight 27'},
    {'site': 'KSC LC 39A', 'coordinates': [28.573255, -80.646895], 'class': 0, 'flight': 'Flight 40'},
    {'site': 'VAFB SLC 4E', 'coordinates': [34.632834, -120.610745], 'class': 1, 'flight': 'Flight 22'},
    {'site': 'VAFB SLC 4E', 'coordinates': [34.632834, -120.610745], 'class': 0, 'flight': 'Flight 36'}
]

# 3. Create a Marker Cluster object to cleanly bundle overlapping pins at the same site
marker_cluster = MarkerCluster().add_to(outcome_map)

# 4. Loop through the records to assign colors and add them to the cluster group
for record in launch_records:
    # Assign green for success (1) and red for failure (0)
    marker_color = 'green' if record['class'] == 1 else 'red'
    icon_type = 'check' if record['class'] == 1 else 'times'
    
    folium.Marker(
        location=record['coordinates'],
        popup=f"<b>Site:</b> {record['site']}<br><b>Outcome:</b> {'Success' if record['class']==1 else 'Failure'}<br>{record['flight']}",
        icon=folium.Icon(color=marker_color, icon=icon_type, prefix='fa')
    ).add_to(marker_cluster)

# 5. Display the generated outcome map
outcome_map


After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
